In [ ]:
import os
from pathlib import Path
import pandas as pd
import time
from datetime import datetime
from sklearn.linear_model import LinearRegression
import numpy as np
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from src.analyse.utils import get_straight_line_distance, wrangle_actuals, normalize_dates_for_lr

load_dotenv()

In [ ]:
calculate_distance = False
save_figs = False

In [ ]:
today = pd.Timestamp.now(tz="UTC")
today

In [ ]:
output_dir = Path(f"outputs/nb03_analyse_listings_{datetime.now().strftime('%Y%m%d-%H%M%S')}")
if save_figs:
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
input_dir = Path('outputs/funda/')
files = sorted(list(input_dir.glob('*.csv')))
files.remove(input_dir / 'extra_info.csv')
files

In [ ]:
df_extra = pd.read_csv(input_dir / 'extra_info.csv')
df_extra.tail()

In [ ]:
# wrangle extra info
df_extra["vve"] = df_extra["vve"].fillna(df_extra["vve"].median()) # add median where VvE is missing

df_extra.tail()

In [ ]:
ind = -1
df_raw = pd.read_csv(files[ind])
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
df = df_raw.copy()
df["publish_date"] = pd.to_datetime(df_raw["publish_date"], format="mixed", utc=True)
df["year_quarter"] = df["publish_date"].dt.to_period("Q")
df["year_month"] = df["publish_date"].dt.to_period("M")
df["price_per_sqm"] = df["price"] / df["floor_area"]
df["size_bracket"] = pd.cut(
    df["floor_area"],
    bins=[0, 50, 80, 110, np.inf],
    labels=["Small", "Medium", "Large", "Extra Large"]
)
df["status"] = df["status"].apply(lambda x: x if x not in ["none"] else "available")
df["time_on_market_days"] = (today - df["publish_date"]).dt.days
df["time_on_market_days"] = df["time_on_market_days"].where(df["status"] == "available", None)
df["time_on_market_months"] = df["time_on_market_days"] / 30.44
df.sort_values("publish_date", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

In [ ]:
df.publish_date.min().date(), df.publish_date.max().date()

In [ ]:
# The latest entry
df.sort_values("publish_date").iloc[-1]

In [ ]:
energy_mapping = {
    'A++++': 11,
    'A+++': 10,
    'A++': 9,
    'A+': 8,
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 3,
    'F': 2,
    'G': 1
}

df['energy_label_numeric'] = df['energy_label'].map(energy_mapping)


In [ ]:
df = pd.merge(df, df_extra, on="address", how="left")
df.head()

In [ ]:
df["extra_score"] = df[["floor_done", "balcony_score", "bathtub", "bathroom_renovated", "kitchen_renovated", "own_land"]].sum(axis=1)
df["workload_score"] = df[["floor_done", "bathroom_renovated", "kitchen_renovated"]].sum(axis=1)
df.tail()

In [ ]:
df_actuals = None
try:
    df_actuals = pd.read_csv(input_dir / "actuals_extracted.csv")
except FileNotFoundError:
    print("Actuals file not found. Please provide the file or check the path.")

df_actuals.head()

In [ ]:
if df_actuals is not None:
    df_actuals = wrangle_actuals(df_actuals)
    display(df_actuals.head())

In [ ]:
df_dists = None

In [ ]:
reference_address = os.getenv("REFERENCE_ADDRESS", "Amsterdam Central Station")
if calculate_distance:
    print(f"Reference address: {reference_address}")

    distances = []
    for ind, row in df.iterrows():
        print(f"row {ind}", end="\r")
        address = row["address"]
        dist = get_straight_line_distance(address, reference_address)
        dist["id"] = row["id"]
        distances.append(dist)

        time.sleep(0.2)

    df_dists = pd.DataFrame(distances)
    display(df_dists.head())

    df = pd.merge(df, df_dists, on="id")
    df.head()


In [ ]:
if df_dists is not None and save_figs:
    df_dists.to_csv(output_dir / f"distances_from_{reference_address.replace(' ', '_')}.csv")
    print("Saved dists dataframe")
elif df_dists is None:
    print("Try reading the distances dataframe from a file.")
    try:
        file_name = f"distances_from_{reference_address.replace(' ', '_')}.csv"
        df_dists = pd.read_csv(input_dir / file_name)
        print(f"Distances dataframe loaded from file {file_name}.")
        df = pd.merge(df, df_dists, on="id")
        print("Merged distances into main dataframe.")
    except FileNotFoundError:
        print("Distances file not found. Please calculate distances or provide the file.")

In [ ]:
df_dists.head()

In [ ]:
df_grouped = df.groupby("year_month").agg(
    num_listings=("id", "count"),
    price_avg=("price", "mean"),
    price_median=("price", "median"),
    price_per_sqm=("price_per_sqm", "mean"),
    living_area=("floor_area", "mean"),
)
df_grouped.reset_index(inplace=True)
df_grouped["year_month"] = df_grouped["year_month"].dt.to_timestamp()
df_grouped.sort_values("year_month", inplace=True)
df_grouped.dtypes

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.violinplot(
    data=df,
    x="number_of_bedrooms",
    y="floor_area",
    inner="quartile",
)

means = df.groupby("number_of_bedrooms")["floor_area"].mean()
for i, (category, mean_val) in enumerate(means.items()):
    plt.text(i, mean_val, f'{mean_val:.1f}', 
            ha='center', va='bottom', fontweight='bold', color='b')

floor_avg = df["floor_area"].mean()
plt.axhline(
    y=floor_avg,
    xmin=-0.5,
    xmax=len(df_grouped) - 0.5,
    color="red",
    linestyle="dashed",
    label=f"Overall Average ({floor_avg:.1f} m2)",
)

plt.legend()
plt.title("Floor Area by Number of Bedrooms")
plt.xlabel("Number of Bedrooms")
plt.ylabel("Floor Area (sqm)")

if save_figs:
    plt.savefig(output_dir / "floor_area_violinplot.png", bbox_inches="tight")

plt.show()

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="number_of_bedrooms",
    y="floor_area",
    width=0.3,
    showmeans=True,
    # meanprops={"marker": "o", "markerfacecolor": "red", "markersize": 8},
    # boxprops={"alpha": 0.5},
)

floor_avg = df["floor_area"].mean()
plt.axhline(
    y=floor_avg,
    xmin=-0.5,
    xmax=len(df_grouped) - 0.5,
    color="red",
    linestyle="dashed",
    label=f"Overall Average ({floor_avg:.1f} m2)",
)

plt.legend()
plt.title("Floor Area by Number of Bedrooms")
plt.xlabel("Number of Bedrooms")
plt.ylabel("Floor Area (sqm)")

if save_figs:
    plt.savefig(output_dir / "floor_area_violinplot.png", bbox_inches="tight")

plt.show()

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="size_bracket",
    y="floor_area",
    showmeans=True,
)

floor_avg = df["floor_area"].mean()
plt.axhline(
    y=floor_avg,
    xmin=-0.5,
    xmax=len(df["size_bracket"].cat.categories) - 0.5,
    color="red",
    linestyle="dashed",
    label=f"Overall Average ({floor_avg:.1f} m2)",
)

plt.xticks(rotation=45, ha="right")
plt.legend()
plt.title("Floor Area by Size Bracket")
plt.xlabel("Size Bracket")
plt.ylabel("Floor Area (sqm)")

if save_figs:
    plt.savefig(output_dir / "floor_area_boxplot_by_size_bracket.png", bbox_inches="tight")

plt.show()

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="size_bracket",
    y="price_per_sqm",
    showmeans=True,
)

price_per_sqm_avg = df["price_per_sqm"].mean()
plt.axhline(
    y=price_per_sqm_avg,
    xmin=-0.5,
    xmax=len(df["size_bracket"].cat.categories) - 0.5,
    color="red",
    linestyle="dashed",
    label=f"Overall Average ({price_per_sqm_avg:.1f} €/m2)",
)

plt.xticks(rotation=45, ha="right")
plt.legend()
plt.title("Ask Price per Square Meter by Size Bracket")
plt.xlabel("Size Bracket")
plt.ylabel("Price per Square Meter (€/m2)")

if save_figs:
    plt.savefig(output_dir / "ask_price_per_sqm_boxplot_by_size_bracket.png", bbox_inches="tight")

plt.show()

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="year_quarter",
    y="price_per_sqm",
    hue="size_bracket",
    showmeans=True,
    palette="Set2",
)

means = df.groupby(["year_quarter", "size_bracket"])["price_per_sqm"].mean().reset_index()
means["year_quarter"] = means["year_quarter"].astype(str)
sns.lineplot(
    data=means,
    x="year_quarter",
    y="price_per_sqm",
    hue="size_bracket",
    marker="o",
    palette="Set2",
)

plt.grid()
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.title("Ask Price per sqm by Year Quarter")
plt.xlabel("Year Quarter")
plt.ylabel("Ask Price per sqm (€/m2)")

if save_figs:
    plt.savefig(output_dir / "ask_price_per_sqm_boxplot_by_year_quarter.png", bbox_inches="tight")

plt.show()

In [ ]:
def plot_price_per_sqm_over_time(df):
    _ = plt.figure(figsize=(12, 6))

    sns.boxplot(
        data=df,
        x="year_month",
        y="price_per_sqm",
        showmeans=True,
    )

    avg = df["price_per_sqm"].mean()
    plt.axhline(
        y=avg,
        color="red",
        linestyle="dashed",
        label=f"Overall Average ({avg:,.0f} €/m²)",
    )

    plt.title("Average Ask Price per Square Meter Over Time")
    plt.xticks(rotation=45, ha="right")
    plt.legend()

    if save_figs:
        plt.savefig(output_dir / "ask_price_per_sqm_over_time.png", bbox_inches="tight")

    plt.show()

In [ ]:
plot_price_per_sqm_over_time(df)

In [ ]:
df.head()

In [ ]:
df.status.unique()

In [ ]:
def filter_df(df, match_criteria):
    df_filtered = df.copy()
    for col, criteria in match_criteria.items():
        if isinstance(criteria, tuple) and len(criteria) == 2 and all(isinstance(c, (int, float)) for c in criteria):
            df_filtered = df_filtered[(df_filtered[col] >= criteria[0]) & (df_filtered[col] <= criteria[1])]
        elif isinstance(criteria, tuple):
            df_filtered = df_filtered[df_filtered[col].isin(criteria)]
    return df_filtered

In [ ]:
match_criteria = {
    "size_bracket": ("Medium", "Large"),
    "number_of_bedrooms": (1, 4),
    # "energy_label": ("D", "C", "B", "A", "A+", "A++"),
    "status": ("available", "under_bid", "sold_under_reservation", "sold"),
    # "status": ("available", "under_bid"),
}

df_filtered = filter_df(df, match_criteria)
df_filtered.head()

In [ ]:
plot_price_per_sqm_over_time(df_filtered)

In [ ]:
cols_of_interest = [
    "address", "price_per_sqm",
    "price", "floor_area", "number_of_bedrooms",
    "distance_km", "estimated_walking_min",
    "energy_label", "status", "url", "publish_date",
    "time_on_market_months", "extra_score", "workload_score",
    "size_bracket"
]
cols_not_available = [col for col in cols_of_interest if col not in df_filtered.columns]
for col in cols_not_available:
    cols_of_interest.remove(col)
    print(f"Column '{col}' not found in dataframe, skipping it.")

df_filtered[cols_of_interest]

In [ ]:
df_filtered[cols_of_interest].sort_values("price_per_sqm")

In [ ]:
criteria_available = {
    "status": ("available", "under_bid"),
}
df_available = filter_df(df_filtered, criteria_available)
df_available

In [ ]:
criteria = {
    "status": ("available", "under_bid"),
    "price": (0, 400000),
    "size_bracket": ("Large"),
}
df_available = filter_df(df, criteria)
if save_figs:
    df_available.to_csv(output_dir / "df_available.csv", index=False)
    print("Saved available listings dataframe")
df_available.sort_values("price_per_sqm")

In [ ]:
def plot_price_per_sqm_floor_area_scatter(df, size_col="extra_score", hue_col="workload_score", palette=None, title="Price per Square Meter vs. Floor Area", save_figs=False):
    fig = plt.figure(figsize=(8,8))
    sns.scatterplot(
        data=df,
        x="floor_area",
        y="price_per_sqm",
        size=size_col,
        hue=hue_col,
        palette=palette
    )
    floor_avg = df["floor_area"].mean()
    plt.axvline(
        x=floor_avg,
        label=f"avg floor area ({floor_avg:,.0f})"
    )
    price_sqm_avg = df["price_per_sqm"].mean()
    plt.axhline(
        price_sqm_avg, label=f"avg price per sqm ({price_sqm_avg:,.0f})"
    )

    plt.grid()
    plt.legend()
    plt.title(title)
    plt.legend(loc="upper right", bbox_to_anchor=(1.4, 1))
    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_vs_floor_area.png", bbox_inches="tight")
    plt.show()

In [ ]:
plot_price_per_sqm_floor_area_scatter(df_available, title="Price per Square Meter vs. Floor Area For Available & Under Bid", save_figs=save_figs)

In [ ]:
criteria = {
    "status": ("under_bid", "sold_under_reservation", "sold"),
    "floor_area": (75, 110),
}
df_sold = filter_df(df, criteria)
plot_price_per_sqm_floor_area_scatter(
    df_sold,
    size_col="estimated_walking_min",
    hue_col="year_month",
    palette="viridis",
    title="Price per Square Meter vs. Floor Area For Sold & Under Reservation", 
    save_figs=save_figs
)

In [ ]:
if df_actuals is not None:
    plot_price_per_sqm_floor_area_scatter(
        df_actuals,
        size_col="price",
        hue_col="year_quarter",
        palette="viridis",
        title="Price per Square Meter vs. Floor Area For Actual Sales",
        save_figs=save_figs
    )

In [ ]:
_ = plt.figure(figsize=(10,10))
sns.scatterplot(
    data=df,
    x="floor_area",
    y="price_per_sqm",
    hue="year_month",
    palette="viridis",
)

plt.grid()
plt.title("Price per sqm by Floor Area and Year Month")
plt.xlabel("Floor Area (m²)")
plt.ylabel("Price per sqm (€)")

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_by_floor_area_and_year_month.png", bbox_inches="tight")

## LR to predict price per sqm in the area of actuals

In [ ]:
def fit_lr_to_predict_price_per_sqm(df, date_col="date"):
    # Convert date to numeric (months since epoch)
    X = normalize_dates_for_lr(df, date_col=date_col)

    y = df['price_per_sqm'].values

    # Fit the model
    model = LinearRegression()
    model.fit(X, y)

    # Print model parameters
    print(f"Coefficient (slope): {model.coef_[0]:.2f} €/m² per month")
    print(f"Intercept: {model.intercept_:.2f} €/m²")
    print(f"R² score: {model.score(X, y):.3f}")
    return model

In [ ]:
normalize_dates_for_lr(pd.DataFrame({"date": pd.to_datetime(["2000-01-01", "2020-01-01"]).to_period("M").to_timestamp()}), date_col="date")

In [ ]:
def plot_price_per_sqm_over_time(df, model: LinearRegression = None, save_figs=False) -> plt.Figure:
    fig = plt.figure(figsize=(12, 6))

    sns.lineplot(
        data=df,
        x="year_month",
        y="price_per_sqm",
        marker="o",
    )

    avg = df["price_per_sqm"].mean()
    plt.axhline(
        y=avg,
        color="blue",
        linestyle="dashed",
        alpha=0.7,
        label=f"Overall Average ({avg:,.0f} €/m²)",
    )
    if model is not None:
        X = normalize_dates_for_lr(df, date_col="year_month")
        y_pred = model.predict(X)
        plt.plot(df["year_month"], y_pred, 'r-', linewidth=2, label='Linear Fit')

    plt.title("Price per Square Meter Over Time For Actual Sales")
    plt.ylim(0, plt.ylim()[1])
    plt.xlabel("Year-Month")
    plt.ylabel("Price per Square Meter")
    plt.grid()
    plt.tight_layout()
    plt.title("Average Price per Square Meter Over Time")
    # plt.xticks(rotation=45, ha="right")
    plt.legend()

    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_over_time.png", bbox_inches="tight")

    return fig

In [ ]:
if df_actuals is not None:
    plot_price_per_sqm_over_time(df_actuals, model=fit_lr_to_predict_price_per_sqm(df_actuals), save_figs=save_figs)

In [ ]:
if df_actuals is not None:
    df_actuals_filtered = df_actuals[df_actuals["year_quarter"] > "2019Q4"]
    df_actuals_filtered = df_actuals_filtered[df_actuals_filtered["floor_area"] < 100]
    model = fit_lr_to_predict_price_per_sqm(df_actuals_filtered)
    plot_price_per_sqm_over_time(df_actuals_filtered, model=model, save_figs=save_figs)

In [ ]:
dates_of_interest = ["2025-01", "2025-06", "2025-12", "2026-02", "2026-03", "2026-06", "2026-12", "2029-01"]
if model is not None:
    df_dates_of_interest = pd.DataFrame({"date": pd.to_datetime(dates_of_interest).to_period("M").to_timestamp()})
    X = normalize_dates_for_lr(df_dates_of_interest, date_col="date")
    y_pred = model.predict(X)

    X_today = normalize_dates_for_lr(pd.DataFrame({"date": [pd.Timestamp.now()]}), date_col="date")
    y_pred_today = model.predict(X_today)

    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df_actuals_filtered,
        x="year_month",
        y="price_per_sqm",
        hue="size_bracket",
        marker="o",
        palette="tab10"
    )

    plt.plot(df_dates_of_interest["date"], y_pred, color="red", linewidth=2, label='Linear Fit')
    plt.axvline(today, color="gray", linestyle="dashed", label=f"Today ({y_pred_today[0]:,.0f} €/m²)")
    plt.xlabel("Date")
    plt.ylabel("Price per Square Meter")
    plt.title("Price per Square Meter Over Time with Future Predictions")
    plt.legend()
    plt.grid()

## Merge actuals and asks

In [ ]:
assert df_actuals is not None, "Actuals dataframe is not available. Please provide the actuals data to plot price per sqm over time."

In [ ]:
df_actuals["type"] = "actual"
df["type"] = "listing"

cols_shared = list(set(df_actuals.columns) & set(df.columns))
cols_shared

In [ ]:
df_combined = pd.concat([df_actuals[cols_shared], df[cols_shared]], ignore_index=True)
ts_lambda = lambda x: x.to_timestamp() if isinstance(x, pd.Period) else x
df_combined["year_quarter"] = df_combined["year_quarter"].apply(ts_lambda)
df_combined["year_month"] = df_combined["year_month"].apply(ts_lambda)
df_combined.head()

In [ ]:
mask_actuals = (df_combined["type"] == "actual") & (df_combined["year_quarter"] > pd.Timestamp("2019-12-31"))
model_actuals = fit_lr_to_predict_price_per_sqm(df_combined[mask_actuals], date_col="year_month")

models_listing = {}
for sb in df_combined["size_bracket"].unique():
    print(f"== Train LR for size bucket {sb}")
    mask_listings = (df_combined["type"] == "listing") & (df_combined["year_quarter"] > pd.Timestamp("2019-12-31")) & (df_combined["size_bracket"].isin([sb]))
    model_listings = fit_lr_to_predict_price_per_sqm(df_combined[mask_listings], date_col="year_month")
    models_listing[sb] = model_listings

In [ ]:
dates_of_interest = ["2025-01", "2025-06", "2025-12", "2026-02", "2026-03", "2026-06", "2026-12"]
df_dates_of_interest = pd.DataFrame({"date": pd.to_datetime(dates_of_interest).to_period("M").to_timestamp()})
X = normalize_dates_for_lr(df_dates_of_interest, date_col="date")
y_act = model_actuals.predict(X)
Y_list = {}
for sb, model in models_listing.items():
    Y_list[sb] = model.predict(X)
y_list_m = Y_list.get("Medium", None)
y_list_l = Y_list.get("Large", None)

X_today = normalize_dates_for_lr(pd.DataFrame({"date": [pd.Timestamp.now()]}), date_col="date")
y_act_today = model_actuals.predict(X_today)[0]
y_list_m_today = models_listing.get("Medium").predict(X_today)[0]
y_list_l_today = models_listing.get("Large").predict(X_today)[0]

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_combined[df_combined["year_quarter"] > pd.Timestamp("2019-12-31")],
    x="year_month",
    y="price_per_sqm",
    hue="type",
    style="size_bracket",
    marker="o",
    palette="tab10"
)

# Plot extrapolations
plt.plot(df_dates_of_interest["date"], y_act, color="red", linewidth=2, ls='--', label='LR actuals')
plt.plot(df_dates_of_interest["date"], y_list_m, color="blue", linewidth=2, label='LR listings (Medium)')
plt.plot(df_dates_of_interest["date"], y_list_l, color="green", linewidth=2, label='LR listings (Large)')
plt.axvline(today, color="gray", linestyle="dashed", label=f"Today: act {y_act_today:.0f}, ask (M) {y_list_m_today:.0f}, (L) {y_list_l_today:.0f} €/m²")

# Plot listing averages
x_listin_min = df_combined[df_combined["type"] == "listing"]["year_month"].min()
x_listin_max = df_combined[df_combined["type"] == "listing"]["year_month"].max()

mask = df_combined["type"] == "listing"
means = df_combined[mask].groupby("size_bracket")["price_per_sqm"].mean().reset_index()
linestyles = {
    "Small": "dashed",
    "Medium": "dotted",
    "Large": "dashdot",
    "Extra Large": (0, (3, 1, 1, 1)),
}
for _, row in means.iterrows():
    sb = row["size_bracket"]
    mean_price_per_sqm = row["price_per_sqm"]
    plt.plot(
        [x_listin_min, x_listin_max],
        [mean_price_per_sqm]*2, color="orange",
        linestyle=linestyles.get(sb, "dashed"),
        label=f"Avg price {sb} ({mean_price_per_sqm:.0f} €/m²)")

plt.xlabel("Date")
plt.ylabel("Price per Square Meter")
plt.title("Price per Square Meter Over Time with Future Predictions")
plt.legend()
plt.grid()

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_over_time_with_predictions.png", bbox_inches="tight")
plt.show()

## Estimate an actual market price of subgroup today

1. choose subgroup with largest amount of data (assume that groups develop in parallel)
2. ground starting point to latest truth, i.e. actuals
3. apply the development of the listing price to extrapolate until today (assume, that listing price goes hand in hand with actuals)
4. apply the difference between modelled subgroup and your subgroup of interest to adjust the estimated market price for your subgroup of interest
5. compare the estimated market price for your subgroup of interest to the listing prices of your subgroup of interest today


In [ ]:
size_brackets = list(df["size_bracket"].cat.categories)
size_brackets

In [ ]:
ind_of_interest = 2
sb_interest = size_brackets[ind_of_interest]

# 1. choose subgroup with largest amount of data (assume that groups develop in parallel)

data_actuals_filtered = df_actuals[df_actuals["year_quarter"] > "2023Q4"]
data_of_recent_actuals_per_bucket = data_actuals_filtered.groupby("size_bracket").count()["date"]
data_of_recent_actuals_per_bucket

In [ ]:
sb_chosen = data_of_recent_actuals_per_bucket.idxmax()
sb_chosen

In [ ]:
# 2. ground starting point to latest truth, i.e. actuals
data_actuals_filtered = data_actuals_filtered[data_actuals_filtered["size_bracket"] == sb_chosen]
price_per_sqm_ref = data_actuals_filtered.groupby("size_bracket")["price_per_sqm"].mean()[sb_chosen]
price_per_sqm_ref

In [ ]:
# 3. apply the development of the listing price to extrapolate until today (assume, that listing price goes hand in hand with actuals)
date_of_latest_actual = data_actuals_filtered["year_quarter"].max()
months_since_latest_actual = (pd.Timestamp.now().to_period("M") - date_of_latest_actual.to_period("M")).n
date_of_latest_actual, months_since_latest_actual

In [ ]:
price_diff_since_last_actual = {
    sb: models_listing[sb].coef_[0] * months_since_latest_actual
    for sb in [sb_chosen, sb_interest]
}
price_diff_since_last_actual # € / m2

In [ ]:
current_market_price_estimated = {
    sb: price_per_sqm_ref + price_diff_since_last_actual[sb]
    for sb in [sb_chosen, sb_interest]
}
current_market_price_estimated

In [ ]:
# 4. apply the difference between modelled subgroup and your subgroup of interest to adjust the estimated market price for your subgroup of interest
df_avg_listing_prices_per_bracket = df_combined[df_combined["type"] == "listing"].groupby("size_bracket")["price_per_sqm"].mean()
subgroup_diff = df_avg_listing_prices_per_bracket[sb_interest] - df_avg_listing_prices_per_bracket[sb_chosen]
subgroup_diff

In [ ]:
current_market_price_of_interest_estimated = {
    sb: current_market_price_estimated[sb] + subgroup_diff
    for sb in [sb_chosen, sb_interest]
}
current_market_price_of_interest_estimated # EUR / m2

In [ ]:
date_of_latest_actual

In [ ]:
# 5. compare the estimated market price for your interest group to the listing prices of your subgroup of interest today for a sanity check
mask = (df_combined["type"] == "listing") & (df_combined["size_bracket"] == sb_interest)
fig = plot_price_per_sqm_over_time(df_combined[mask], save_figs=False)

sns.scatterplot(
    data=data_actuals_filtered,
    x="year_month",
    y="price_per_sqm",
    marker="s",
    zorder=3,
    color="orange",
    label=f"Actuals for {sb_chosen}"
)
plt.scatter(
    x=date_of_latest_actual,
    y=price_per_sqm_ref,
    color="orange",
    label=f"Estimated market price for {sb_chosen} at latest actual ({price_per_sqm_ref:.0f} €/m²)",
    zorder=5
)
plt.plot(
    [date_of_latest_actual, pd.Timestamp.now()],
    [price_per_sqm_ref, current_market_price_estimated[sb_chosen]],
    color="orange",
    alpha=0.5,
    linestyle="dashed",
    label=f"Development of listing prices for {sb_chosen} since ({models_listing[sb_chosen].coef_[0]:.2f} €/m² per month)",
    zorder=4
)
plt.plot(
    [date_of_latest_actual, pd.Timestamp.now()],
    [price_per_sqm_ref, current_market_price_estimated[sb_interest]],
    color="blue",
    alpha=0.5,
    linestyle="dashed",
    label=f"Development of listing prices for {sb_interest} since ({models_listing[sb_interest].coef_[0]:.2f} €/m² per month)",
    zorder=4
)

plt.plot(
    [pd.Timestamp.now()]*2,
    [current_market_price_estimated[sb_interest], current_market_price_of_interest_estimated[sb_interest]],
    color="blue",
    alpha=0.5,
    linestyle="-.",
    label=f"Price correction between {sb_chosen} and {sb_interest} (from listings, {subgroup_diff:.2f} €/m²)",
    zorder=4
)
for sb, ls, mk in zip([sb_chosen, sb_interest], ["dashed", "dashdot"], ["o", "D"]):
    plt.scatter(
        x=pd.Timestamp.now(),
        y=current_market_price_of_interest_estimated[sb],
        color="blue",
        marker=mk,
        label=f"Estimated current market price for {sb_interest} following development of {sb} ({current_market_price_of_interest_estimated[sb]:.0f} €/m²)",
        zorder=5
    )

plt.xlim(
    min(data_actuals_filtered["year_month"].min(), min(df_combined[mask]["year_month"].min(), date_of_latest_actual)) - pd.DateOffset(months=1), 
    plt.xlim()[1]
)
plt.ylim(plt.ylim()[0], max(df_combined[mask]["price_per_sqm"].max(), current_market_price_of_interest_estimated[sb_interest]) * 1.1)
plt.title(f"Ask Price per Square Meter Over Time for {sb_interest} Listings")

plt.legend(loc="lower right")

if save_figs:
    plt.savefig(output_dir / f"price_per_sqm_over_time_with_estimation_for_{sb_interest}.png", bbox_inches="tight")
plt.show()

## Try to match actuals with listings for price difference

! This would require a lot of data to work

In [ ]:
df_actuals_grouped = df_actuals.groupby(["year_quarter", "size_bracket"]).agg(
    price_per_sqm_avg=("price_per_sqm", "mean"),
    num_sales=("date", "count")
).reset_index()
df_actuals_grouped["year"] = df_actuals_grouped["year_quarter"].dt.year

df_actuals_grouped.head()

In [ ]:
df_asked_grouped = df_sold.groupby(["year_quarter", "size_bracket"]).agg(
    price_per_sqm_avg=("price_per_sqm", "mean"),
    num_sales=("price", "count")
).reset_index()
df_asked_grouped["year_quarter"] = df_asked_grouped["year_quarter"].dt.to_timestamp()
df_asked_grouped["year"] = df_asked_grouped["year_quarter"].dt.year

df_asked_grouped.head()

In [ ]:
cols_oi = ["year_quarter", "price_per_sqm_avg", "num_sales", "size_bracket"]
df_actuals_vs_asked =pd.merge(df_actuals_grouped[cols_oi], df_asked_grouped[cols_oi], on=["year_quarter", "size_bracket"], suffixes=("_actuals", "_asked"), how="outer")
df_actuals_vs_asked.tail()

In [ ]:
df_actuals_vs_asked["price_diff"] = df_actuals_vs_asked["price_per_sqm_avg_actuals"] - df_actuals_vs_asked["price_per_sqm_avg_asked"]
df_actuals_vs_asked.tail(20)

In [ ]:
df_to_plot = df_actuals_vs_asked.dropna(subset=["price_diff"])
sns.barplot(df_to_plot, x="year_quarter", y="price_diff", hue="size_bracket", palette="Set2")
plt.title("Difference in Average Price per sqm Between Actual Sales and Asked Prices")
plt.xlabel("Year Quarter")
plt.ylabel("€/m² Difference (Actuals - Asked)")
plt.show()

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="status",
    y="time_on_market_weeks",
    showmeans=True,
)

time_on_market_avg = df["time_on_market_weeks"].mean()
plt.axhline(
    y=time_on_market_avg,
    xmin=-0.5,
    xmax=len(df["status"]) - 0.5,
    color="red",
    linestyle="dashed",
    label=f"Overall Average ({time_on_market_avg:.1f} weeks)",
)

plt.legend()
plt.title("Time on Market by Status")
plt.xlabel("Status")
plt.ylabel("Time on Market (weeks)")

if save_figs:
    plt.savefig(output_dir / "time_on_market_boxplot_by_status.png", bbox_inches="tight")

plt.show()

## Predict listing price to learn value of separate elements

In [ ]:
df_with_extra_info = df[~df.floor.isna()]
df_with_extra_info.head()

In [ ]:
feature_cols = [
    "floor_area", "number_of_bedrooms", "energy_label_numeric",
    "floor_done", 
    # "extra_score", 
    # "workload_score",
    # "bathroom_renovated", "kitchen_renovated",
    # "vve",
    # "floor", "own_land"
]
y_col = "price_per_sqm"

In [ ]:
def univariate_analysis(df, feature_cols, y_col):
    results = []
    for feature in feature_cols:
        mask = ~df[[feature, y_col]].isna().any(axis=1)
        X = df.loc[mask, feature]
        y = df.loc[mask, y_col]
        
        corr, p_value = stats.pearsonr(X, y)
        
        # For binary features, compare group means
        if df[feature].nunique() <= 2:
            groups = df.groupby(feature)[y_col].agg(['mean', 'count'])
            if len(groups) == 2:
                diff = groups.loc[1, 'mean'] - groups.loc[0, 'mean']
                group0 = df[df[feature] == 0][y_col].dropna()
                group1 = df[df[feature] == 1][y_col].dropna()
                t_stat, t_pval = stats.ttest_ind(group1, group0)
            else:
                diff, t_pval = np.nan, np.nan
        else:
            diff, t_pval = np.nan, np.nan
            
        results.append({
            'feature': feature,
            'correlation': corr,
            'p_value': p_value,
            'mean_difference': diff,
            'ttest_pvalue': t_pval,
            'n_obs': len(X)
        })
    
    return pd.DataFrame(results).sort_values('correlation', ascending=False, key=abs)


In [ ]:
univariate_results = univariate_analysis(df_with_extra_info, feature_cols, y_col)
univariate_results

In [ ]:
# The features with a small p-value, like <0.2? Keep them and do this:
X = df_with_extra_info[feature_cols].replace([np.inf, -np.inf, np.nan], 0.5)
model_listing = LinearRegression().fit(X, df_with_extra_info[y_col])

effects = {
    feature_cols[i]: model_listing.coef_[i] for i in range(len(feature_cols))
}
effects["base_price_per_sqm"] = model_listing.intercept_
effects

In [ ]:
predicted_prices = model_listing.predict(X)
df_with_extra_info["predicted_price_per_sqm"] = predicted_prices
df_with_extra_info["price_predicted"] = df_with_extra_info["predicted_price_per_sqm"] * df_with_extra_info["floor_area"]
df_with_extra_info["diff"] = df_with_extra_info["price_per_sqm"] - df_with_extra_info["predicted_price_per_sqm"]
df_with_extra_info["mae"] = np.abs(df_with_extra_info["diff"])
df_with_extra_info[["price_per_sqm", "predicted_price_per_sqm", "price", "price_predicted", "diff", "mae"]].head()

In [ ]:
df_with_extra_info

In [ ]:
plt.figure(figsize=(10, 6))
df_to_plot = df_with_extra_info.melt(id_vars=["floor_area"], value_vars=["price_per_sqm", "predicted_price_per_sqm"], var_name="type", value_name="price_m2")

sns.scatterplot(
    data=df_to_plot,
    x="floor_area",
    y="price_m2",
    hue="type",
    palette="Set2",
    alpha=0.7
)

text_content = "Impacts:\n" + "\n".join([f"  {k + ':':<24}{v:,.1f}" if isinstance(v, (int, float)) else f"  {k + ':':<24}{v}" for k, v in effects.items()])

plt.text(0.02, 0.02, text_content, transform=plt.gca().transAxes,
         fontsize=9, verticalalignment='bottom', family='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.title("Actual and predicted listing price of active listings")
plt.xlabel("Floor Area (sqm)")
plt.ylabel("€ / m²")
plt.grid()

if save_figs:
    plt.savefig(output_dir / "actual_vs_predicted_price_per_sqm.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
df_to_plot = df_with_extra_info[~df_with_extra_info["status"].isin(["sold", "sold_under_reservation"])]
df_to_plot = df_to_plot.melt(id_vars=["floor_area"], value_vars=["price", "price_predicted"], var_name="type", value_name="cost")
sns.scatterplot(
    data=df_to_plot,
    x="floor_area",
    y="cost",
    hue="type",
    palette="Set2",
    alpha=0.7
)

plt.title("Actual and predicted listing price of active listings")
plt.xlabel("Floor Area (sqm)")
plt.ylabel("€")
plt.grid()

if save_figs:
    plt.savefig(output_dir / "actual_vs_predicted_listing_price.png", bbox_inches="tight")
plt.show()

In [ ]:
cols_of_interest

In [ ]:
current_market_price_of_interest_estimated

In [ ]:
market_price_sqm_estimation_col = f"market_price_sqm_estimation_for_{sb_interest.lower()}"
market_price_estimation_col = f"market_price_estimation_if_{sb_interest.lower()}"
df_with_extra_info[market_price_sqm_estimation_col] = current_market_price_of_interest_estimated[sb_interest]
df_with_extra_info[market_price_estimation_col] = current_market_price_of_interest_estimated[sb_interest] * df_with_extra_info["floor_area"]


In [ ]:
ids_of_interest = [287, 633, 969]
columns_of_interest = cols_of_interest + ["predicted_price_per_sqm", "price_predicted", market_price_sqm_estimation_col, market_price_estimation_col]
df_of_interest = df_with_extra_info[df_with_extra_info["id"].isin(ids_of_interest)][columns_of_interest].set_index("address").T

In [ ]:
if save_figs:
    df_of_interest.to_csv(output_dir / "df_of_interest.csv")
df_of_interest

In [ ]:
current_market_price_of_interest_estimated

In [ ]:
for id in ids_of_interest:
    listing = df_with_extra_info[df_with_extra_info["id"] == id]
    price_per_sqm = listing["price_per_sqm"].values[0]
    predicted_price_per_sqm = listing["predicted_price_per_sqm"].values[0]
    price = listing["price"].values[0]
    extra_score = listing["extra_score"].values[0]
    size_bracket = listing["size_bracket"].values[0]
    predicted_price = listing["price_predicted"].values[0]
    area = listing["floor_area"].values[0]
    key = size_bracket if size_bracket in current_market_price_estimated else sb_interest
    price_with_market_estimation = current_market_price_of_interest_estimated[key] * area
    address = listing["address"].values[0]
    bedrooms = listing["number_of_bedrooms"].values[0]

    print(f"Listing: {address}")
    print(f"  Floor area: {area} m²")
    print(f"  Number of bedrooms: {bedrooms}")
    print(f"  Size bracket: {size_bracket}")
    print(f"  Extra score: {extra_score}")
    print(f"  Actual price per sqm: {price_per_sqm:,.2f} €/m²")
    print(f"  Predicted price per sqm: {predicted_price_per_sqm:,.2f} €/m²")
    if sb_interest == size_bracket:
        print(f"  Estimated current market price per sqm for {sb_interest}: {current_market_price_of_interest_estimated[key]:,.2f} €/m²")
    print(f"  Actual total price: {price:,.2f} €")
    print(f"  Predicted total price: {predicted_price:,.2f} €")
    if sb_interest == size_bracket:
        print(f"  Estimated total price using market estimation: {price_with_market_estimation:,.2f} €")
    print()